In [ ]:

#chatbot using RNN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Dataset
questions = [
    "hi", "hello",
    "how are you",
    "what is ai",
    "bye"
]

labels = [
    "greeting", "greeting",
    "status",
    "ai",
    "bye"
]

responses = {
    "greeting": "Hello!",
    "status": "I am fine, how can I help you?",
    "ai": "AI stands for Artificial Intelligence.",
    "bye": "Goodbye!"
}

# Train model
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)

model = LogisticRegression()
model.fit(X, labels)

# Chat function
def chat(msg):
    intent = model.predict(vectorizer.transform([msg]))[0]
    return responses[intent]

# Chat loop
print("Chatbot ready! (type 'exit' to stop)")
while True:
    user = input("You: ")
    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))

In [ ]:
#chatbot using lstn
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset
questions = ["hi", "hello", "how are you", "what is ai", "bye"]
answers = ["hello", "hi", "i am fine", "ai is artificial intelligence", "goodbye"]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(questions + answers)

q_seq = pad_sequences(tokenizer.texts_to_sequences(questions), maxlen=5)
a_seq = pad_sequences(tokenizer.texts_to_sequences(answers), maxlen=5)

# Model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(500, 32, input_length=5),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(5, activation='softmax')  # 5 responses
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Labels (0–4 for answers)
labels = np.array([0, 1, 2, 3, 4])

model.fit(q_seq, labels, epochs=200, verbose=0)

# Chat function
def chat(msg):
    seq = pad_sequences(tokenizer.texts_to_sequences([msg]), maxlen=5)
    pred = model.predict(seq, verbose=0)
    index = np.argmax(pred)
    return answers[index]

# Chat loop
print("LSTM Chatbot Ready! (type 'exit' to stop)")
while True:
    user = input("You: ")
    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))

In [ ]:
!pip install -q transformers==4.41.2 torch

In [ ]:


from transformers import pipeline

qa = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

# 🔥 STRONG CONTEXT (IMPORTANT FIX)
context = """
Artificial Intelligence (AI) is a field that simulates human intelligence in machines.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning is a subset of Machine Learning using neural networks.
Python is a programming language used for AI, ML and data science.
BERT is a transformer-based model used for understanding language.
Azure is a cloud computing platform by Microsoft.
Neural networks are computational models inspired by the human brain.
"""

print("BERT Chatbot Ready (type 'exit' to stop)")

while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Bot: Goodbye!")
        break

    result = qa({
        "question": question,
        "context": context
    })

    print("Bot:", result["answer"])

In [ ]:
!pip install -q transformers sentencepiece

In [ ]:
#chatbot using GPT
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Predefined correct answers (VERY IMPORTANT)
fixed_answers = {
    "what is ai": "Artificial Intelligence is the simulation of human intelligence in machines.",
    "what is python": "Python is a programming language used for software development, AI, and data science.",
    "what is java": "Java is a high-level programming language used for building applications.",
    "what is english": "English is a widely spoken international language."
}

def chat(user_input):
    user_input = user_input.lower()

    # Step 1: Check fixed answers
    if user_input in fixed_answers:
        return fixed_answers[user_input]

    # Step 2: Otherwise use model
    prompt = f"Answer clearly: {user_input}"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=60)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

print("Accurate Chatbot Ready! (type 'exit' to stop)")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    print("Bot:", chat(user))

In [ ]:
#chatbot using TRANSFORMERS
!pip install -q transformers torch sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

questions = [
    "hi",
    "how are you",
    "what is ai",
    "what is python",
    "bye"
]

answers = [
    "hello",
    "i am fine",
    "ai is artificial intelligence",
    "python is a programming language",
    "goodbye"
]

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()


for epoch in range(30):
    for q, a in zip(questions, answers):

        input_text = "question: " + q
        target_text = a

        inputs = tokenizer(input_text, return_tensors="pt")
        labels = tokenizer(target_text, return_tensors="pt").input_ids

        outputs = model(
            input_ids=inputs.input_ids,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

print("Training Completed!")

# Chat function
def chat(user_input):
    model.eval()

    input_text = "question: " + user_input
    inputs = tokenizer(input_text, return_tensors="pt")

    outputs = model.generate(inputs.input_ids, max_length=50)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Chat loop
print("Chatbot Ready! (type 'exit' to stop)")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    print("Bot:", chat(user))